In [ ]:
from google.colab import files
files.upload()  # yah

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"adarshgupta07","key":"49728a9ea62ac83c25d4d5f5f613e0ad"}'}

In [ ]:
import os
from google.colab import files

# Define the standard Kaggle configuration directory
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

kaggle_config_path = os.path.join(kaggle_dir, 'kaggle.json')

# Check if kaggle.json exists in the current directory (where files.upload() typically places it)
if os.path.exists('kaggle.json'):
    # If found, move it to the correct Kaggle directory
    os.rename('kaggle.json', kaggle_config_path)
    os.chmod(kaggle_config_path, 600)
    print("kaggle.json moved to {}".format(kaggle_config_path))
else:
    print("Error: 'kaggle.json' not found in the current directory. Please ensure you have uploaded it using `files.upload()` in a previous cell or re-run the upload cell.")


kaggle.json moved to /root/.kaggle/kaggle.json


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# Dataset download karo
!pip install kaggle -q

import os, json
os.makedirs('/root/.config/kaggle', exist_ok=True)
kaggle_creds = {
    "username": "tumhara_username",
    "key": "tumhara_api_key"
}
with open('/root/.config/kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)

!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
100% 25.7M/25.7M [00:00<00:00, 87.8MB/s]



In [ ]:
# Unzip + Load
import zipfile
with zipfile.ZipFile('imdb-dataset-of-50k-movie-reviews.zip', 'r') as zip_ref:
    zip_ref.extractall('imdb_data')

df = pd.read_csv('imdb_data/IMDB Dataset.csv')
print(df.shape)
df.head()

(50000, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
print(df['sentiment'].value_counts())

sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [ ]:
df['review'][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [ ]:
import re

def clean_text(text):
    text = text.lower()                        # Lowercase
    text = re.sub(r'<.*?>', '', text)          # HTML tags hatao
    text = re.sub(r'[^a-zA-Z\s]', '', text)   # Numbers/symbols hatao
    return text

df['review'] = df['review'].apply(clean_text)
df['review'][0]

'one of the other reviewers has mentioned that after watching just  oz episode youll be hooked they are right as this is exactly what happened with methe first thing that struck me about oz was its brutality and unflinching scenes of violence which set in right from the word go trust me this is not a show for the faint hearted or timid this show pulls no punches with regards to drugs sex or violence its is hardcore in the classic use of the wordit is called oz as that is the nickname given to the oswald maximum security state penitentary it focuses mainly on emerald city an experimental section of the prison where all the cells have glass fronts and face inwards so privacy is not high on the agenda em city is home to manyaryans muslims gangstas latinos christians italians irish and moreso scuffles death stares dodgy dealings and shady agreements are never far awayi would say the main appeal of the show is due to the fact that it goes where other shows wouldnt dare forget pretty picture

In [ ]:
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})
print(df['sentiment'].value_counts())

sentiment
1    25000
0    25000
Name: count, dtype: int64


In [ ]:
X = df['review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("X_train shape:", X_train_tfidf.shape)

X_train shape: (40000, 5000)


In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)
print("Done!")

Done!


In [ ]:
y_pred = model.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, y_pred) * 100, "%")
print(classification_report(y_test, y_pred))

Accuracy: 89.3 %
              precision    recall  f1-score   support

           0       0.90      0.88      0.89      4961
           1       0.88      0.91      0.90      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [ ]:
def predict_sentiment(review):
    cleaned = clean_text(review)
    vectorized = tfidf.transform([cleaned])
    prediction = model.predict(vectorized)
    return "😊 Positive!" if prediction[0] == 1 else "😞 Negative!"

# Test karo!
print(predict_sentiment("This movie was absolutely amazing!"))
print(predict_sentiment("Worst movie I have ever seen!"))

😊 Positive!
😞 Negative!
